In [1]:
import pandas as pd

true_df = pd.read_csv("True.csv")
fake_df = pd.read_csv("Fake.csv")

true_df["label"] = 1   # TRUE
fake_df["label"] = 0   # FAKE


In [2]:
import numpy as np
import re
import joblib
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [3]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>


False

In [4]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


In [5]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)

    words = [
        lemmatizer.lemmatize(word)
        for word in text.split()
        if word not in stop_words and len(word) > 2
    ]

    return ' '.join(words)


In [6]:
true_df['content'] = true_df['title'].fillna('') + ' ' + true_df['text'].fillna('')
fake_df['content'] = fake_df['title'].fillna('') + ' ' + fake_df['text'].fillna('')


In [7]:
df = pd.concat([
    true_df[['content', 'label']],
    fake_df[['content', 'label']]
])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)


In [9]:
df['clean_text'] = df['content'].apply(clean_text)

df = df[df['clean_text'].str.len() > 10]


In [10]:
X = df['clean_text']
y = df['label']


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [12]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

joblib.dump(tfidf, 'tfidf_vectorizer.pkl')


['tfidf_vectorizer.pkl']

In [13]:
models = {
    'logistic_regression': LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        n_jobs=-1,
        random_state=42
    ),

    'random_forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        n_jobs=-1,
        random_state=42
    ),

    'svm': SVC(
        kernel='linear',
        probability=True,
        class_weight='balanced',
        random_state=42
    ),

    'decision_tree': DecisionTreeClassifier(
        max_depth=20,
        class_weight='balanced',
        random_state=42
    ),

    'naive_bayes': MultinomialNB(alpha=0.1)
}


In [14]:
model_results = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train_vec, y_train)
    predictions = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test, predictions)
    model_results[name] = accuracy

    joblib.dump(model, f'{name}_model.pkl')

    print(f"Accuracy: {accuracy * 100:.2f}%")


Training logistic_regression...
Accuracy: 99.01%
Training random_forest...
Accuracy: 99.62%
Training svm...
Accuracy: 99.55%
Training decision_tree...
Accuracy: 99.65%
Training naive_bayes...
Accuracy: 95.39%


In [15]:
best_model_name = max(model_results, key=model_results.get)
best_model = models[best_model_name]

with open('best_model.txt', 'w') as file:
    file.write(best_model_name)


In [16]:
y_pred_best = best_model.predict(X_test_vec)

print(classification_report(
    y_test,
    y_pred_best,
    target_names=['Fake News', 'True News']
))


              precision    recall  f1-score   support

   Fake News       1.00      1.00      1.00      4696
   True News       1.00      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [17]:
cm = confusion_matrix(y_test, y_pred_best)

print("Confusion Matrix:")
print("Actual \\ Predicted")
print("Fake   True")
print(cm)


Confusion Matrix:
Actual \ Predicted
Fake   True
[[4676   20]
 [  11 4273]]


In [18]:
print(f"Best Model: {best_model_name}")
print("All models trained and saved successfully!")


Best Model: decision_tree
All models trained and saved successfully!
